# Messy E-Commerce Customer Database Rescue

## The Scenario
The company database crashed, and the backup data is a mess. We need to clean the "Orders" dataset (`ecommerce.csv`).

In [6]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('ecommerce.csv')
df

,id,customer name,email,purchase date,price,trx,shipping state
0,NaN,port beckwith,pbeckwith94@geocities.com,30 Jul 2025,$460.25,548f8041-6317-4388-8dce-fe6de027505c,Argentina
1,689.0,kimmy haborn,khabornj4@google.es,pending,$539.17,NaN,Panama
2,414.0,matthias calcutt,mcalcuttbh@freewebs.com,11/17/2025,$534.17,4a22264d-4c87-4f9d-b4a5-99e343cb768e,Brazil
3,789.0,ALGERNON NIAVES,aniaveslw@wunderground.com,2025-09-20,$345.84,83a9aa51-0a17-4fd8-a7c3-46a87c019f7b,Philippines
4,NaN,kynthia mcquirter,kmcquirter6s@yellowbook.com,unknown,$283.29,29c1f562-82cb-40d1-97aa-a467feeb24d8,Indonesia
...,...,...,...,...,...,...,...
1095,467.0,ALLISTER MELLOWS,amellowscy@vimeo.com,2025-11-06,$127.81,7dac3d2e-cfce-423b-b34c-24c1efa35950,NaN
1096,122.0,carl lamey,clamey3d@php.net,17 Mar 2025,$500.37,a900e75e-5035-4357-824e-d82f8db6a607,Armenia
1097,NaN,CYMBRE COCKRILL,ccockrill8b@joomla.org,04 Jul 2025,$482.79,98d998ca-0b37-4898-ae25-ab85c4d981db,Portugal
1098,NaN,brynna petrie,bpetriec4@nhs.uk,15 Jul 2025,$780.39,39d7652c-fe34-4e66-bf90-b5fb2cfbbf2f,China


## 1. Handling Missing Data
The dataset is riddled with NaNs. We must:
- Use `.dropna()` to remove rows completely missing an Order ID.
- Use `.fillna()` to replace missing shipping states and transaction with "Unknown".

In [8]:
df = df.dropna(subset=['id'])

df['shipping state'] = df['shipping state'].fillna('Unknown')
df['trx'] = df['trx'].fillna('Unknown')

df.head()

,id,customer name,email,purchase date,price,trx,shipping state
1,689.0,kimmy haborn,khabornj4@google.es,pending,$539.17,Unknown,Panama
2,414.0,matthias calcutt,mcalcuttbh@freewebs.com,11/17/2025,$534.17,4a22264d-4c87-4f9d-b4a5-99e343cb768e,Brazil
3,789.0,ALGERNON NIAVES,aniaveslw@wunderground.com,2025-09-20,$345.84,83a9aa51-0a17-4fd8-a7c3-46a87c019f7b,Philippines
5,760.0,hayes kleewein,hkleeweinl3@kickstarter.com,2025-12-30,$275.30,f91b1691-2e28-486d-82eb-1a81cba02f98,Germany
6,97.0,SUSANNE CUMMINS,scummins2o@paypal.com,unknown,$602.31,Unknown,Vietnam


## 2. Handling Duplicates
A glitch caused users to be charged twice. We must use `.duplicated()` to find identical transactions and `.drop_duplicates(keep='first')` to refund/remove the errors.

In [10]:
duplicates = df[df.duplicated()]
print(f"Found {len(duplicates)} duplicate rows.\n")

df = df.drop_duplicates(keep='first')

df.head()

Found 0 duplicate rows.



,id,customer name,email,purchase date,price,trx,shipping state
1,689.0,kimmy haborn,khabornj4@google.es,pending,$539.17,Unknown,Panama
2,414.0,matthias calcutt,mcalcuttbh@freewebs.com,11/17/2025,$534.17,4a22264d-4c87-4f9d-b4a5-99e343cb768e,Brazil
3,789.0,ALGERNON NIAVES,aniaveslw@wunderground.com,2025-09-20,$345.84,83a9aa51-0a17-4fd8-a7c3-46a87c019f7b,Philippines
5,760.0,hayes kleewein,hkleeweinl3@kickstarter.com,2025-12-30,$275.30,f91b1691-2e28-486d-82eb-1a81cba02f98,Germany
6,97.0,SUSANNE CUMMINS,scummins2o@paypal.com,unknown,$602.31,Unknown,Vietnam


## 3. Data Type Conversions
- **Price**: Remove the symbols (e.g., "$", ",") and use `pd.to_numeric()` to turn it into a float.
- **Purchase Date**: Replace “pending” and “unknown” with the previous datetime, convert from messy text into a proper datetime object using `pd.to_datetime()`, and convert the format to “M/D/Y”.

In [14]:
### Price Conversion
if df['price'].dtype == 'O':
    df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

### Purchase Date Conversion
df['purchase date'] = df['purchase date'].replace(['pending', 'unknown'], np.nan)

df['purchase date'] = pd.to_datetime(df['purchase date'], format='mixed', errors='coerce')

# Replace missing values with the previous datetime using ffill()
df['purchase date'] = df['purchase date'].ffill()

df['purchase date'] = df['purchase date'].dt.strftime('%m/%d/%Y')

df.head()

,id,customer name,email,purchase date,price,trx,shipping state,email domain
1,689.0,Kimmy Haborn,khabornj4@google.es,NaN,539.17,Unknown,Panama,google.es
2,414.0,Matthias Calcutt,mcalcuttbh@freewebs.com,11/17/2025,534.17,4a22264d-4c87-4f9d-b4a5-99e343cb768e,Brazil,freewebs.com
3,789.0,Algernon Niaves,aniaveslw@wunderground.com,09/20/2025,345.84,83a9aa51-0a17-4fd8-a7c3-46a87c019f7b,Philippines,wunderground.com
5,760.0,Hayes Kleewein,hkleeweinl3@kickstarter.com,12/30/2025,275.30,f91b1691-2e28-486d-82eb-1a81cba02f98,Germany,kickstarter.com
6,97.0,Susanne Cummins,scummins2o@paypal.com,12/30/2025,602.31,Unknown,Vietnam,paypal.com


## 4. String Manipulation
- **Names**: Fix chaotic capitalization and extra spaces using `.str.title()` and `.str.strip()`.
- **Email**: Extract the email domain using `.str.split()`.

In [15]:
if df['customer name'].dtype == 'O':
    df['customer name'] = df['customer name'].str.strip().str.title()

if df['email'].dtype == 'O':
    df['email domain'] = df['email'].str.split('@').str[1]

df.head()

,id,customer name,email,purchase date,price,trx,shipping state,email domain
1,689.0,Kimmy Haborn,khabornj4@google.es,NaN,539.17,Unknown,Panama,google.es
2,414.0,Matthias Calcutt,mcalcuttbh@freewebs.com,11/17/2025,534.17,4a22264d-4c87-4f9d-b4a5-99e343cb768e,Brazil,freewebs.com
3,789.0,Algernon Niaves,aniaveslw@wunderground.com,09/20/2025,345.84,83a9aa51-0a17-4fd8-a7c3-46a87c019f7b,Philippines,wunderground.com
5,760.0,Hayes Kleewein,hkleeweinl3@kickstarter.com,12/30/2025,275.30,f91b1691-2e28-486d-82eb-1a81cba02f98,Germany,kickstarter.com
6,97.0,Susanne Cummins,scummins2o@paypal.com,12/30/2025,602.31,Unknown,Vietnam,paypal.com
